# HOPE-JEPA-SIGReg — Kaggle run

A self-supervised image model combining:
- **HOPE** (Nested Learning) backbone: each layer = Self-Modifying Titans memory mixer (learned-optimizer neural long-term memory, full BPTT) + Continuum Memory System (FFN modules at staggered `2^k` update frequencies).
- **JEPA on every layer**: predict masked-patch embeddings from context-patch embeddings in latent space; summed across layers = deep/hierarchical JEPA.
- **SIGReg** (LeJEPA): sketched isotropic-Gaussian covariance regularizer — the documented fix for the representation collapse that self-targeting JEPA would otherwise fall into.

Protocol: SSL pretrain (no labels) with `Σ_l JEPA^l + λ·SIGReg`, then freeze the backbone and train a linear probe (the canonical JEPA eval).

**This notebook runs the SIGReg ablation** (with vs without SIGReg) so the collapse-prevention story shows up directly in the probe accuracies.

> **Kaggle:** enable **GPU (T4 x2 or P100)** in *Settings → Accelerator* before running. Internet must be **On** for the `pip install` and CIFAR download (or attach a dataset).

## 1. Setup

In [ ]:
# Clone the repo if running fresh on Kaggle (or upload the project files as a dataset).
# Option A: clone from GitHub (set INTERNET ON)
# !git clone https://github.com/moelanoby/HOPE-JEPA.git /kaggle/working/HOPE-JEPA
# %cd /kaggle/working/HOPE-JEPA

# Option B: the project files are already in the working dir (uploaded as a dataset/notebook input)
import os, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
os.chdir(WORK)
sys.path.insert(0, WORK)
print('cwd:', os.getcwd())
!pip install -q einops pyyaml tqdm  # torch/torchvision already on Kaggle

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Quick correctness check (run before the long job)

Shapes, BPTT gradient connectivity, and the SIGReg collapse-prevention demonstration. ~1 minute on a T4.

In [ ]:
!python -m tests.test_shapes
!python -m tests.test_sigreg
!python -m tests.test_overfit_batch

## 3. SSL pretrain — WITH SIGReg (the main run)

Uses `config/kaggle_t4.yaml`: full CIFAR-10, 4 HOPE layers, d=256, 100 epochs. On a T4 this is the bulk of your 9h weekly quota — if you need it shorter, edit the config's `ssl.epochs` down.

Watch `eff_rank` in the logs: it should stay well above 1 (SIGReg preventing collapse).

In [ ]:
!python train_ssl.py --config config/kaggle_t4.yaml

## 4. SSL pretrain — WITHOUT SIGReg (ablation)

Same model, SIGReg disabled. Expect `eff_rank` to crash toward 1 (collapse).

In [ ]:
!python train_ssl.py --config config/kaggle_t4.yaml --no-sigreg

## 5. Linear probe both checkpoints

Freeze the backbone, train a single linear head on CIFAR-10 labels, report top-1 accuracy. The WITH-SIGReg run should beat the WITHOUT-SIGReg run — that gap is the headline result.

In [ ]:
print('=== Linear probe: WITH SIGReg ===')
!python linear_probe.py --config config/kaggle_t4.yaml --checkpoint runs/kaggle_t4/checkpoints/ssl_final.pt
print()
print('=== Linear probe: WITHOUT SIGReg (ablation) ===')
!python linear_probe.py --config config/kaggle_t4.yaml --checkpoint runs/kaggle_t4_nosigreg/checkpoints/ssl_final.pt

## 6. Results summary

Random baseline for 10-way CIFAR-10 is **10%**. Compare:
- WITH SIGReg probe accuracy
- WITHOUT SIGReg probe accuracy

and the `eff_rank` trajectories from the two SSL runs (`runs/kaggle_t4/ssl_log.jsonl` vs `runs/kaggle_t4_nosigreg/ssl_log.jsonl`). The no-SIGReg run should show rank collapsing while the SIGReg run holds rank high.

In [ ]:
import json, glob, matplotlib.pyplot as plt

def load(path):
    rows = [json.loads(l) for l in open(path)]
    return list(zip(*[(r['step'], r['eff_rank'], r['jepa']) for r in rows]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, path in [('WITH SIGReg', 'runs/kaggle_t4/ssl_log.jsonl'),
                    ('WITHOUT SIGReg', 'runs/kaggle_t4_nosigreg/ssl_log.jsonl')]:
    if not glob.glob(path):
        continue
    steps, er, jepa = load(path)
    axes[0].plot(steps, er, label=label)
    axes[1].plot(steps, jepa, label=label)
axes[0].set_title('effective rank (high = healthy, 1 = collapsed)'); axes[0].legend()
axes[1].set_title('JEPA loss'); axes[1].set_yscale('log'); axes[1].legend()
plt.tight_layout(); plt.show()